# 04 — Collaborative Filtering Model Training (Sprout-Style)

Train a denoising autoencoder on user rating patterns:
- **Input**: Sparse user rating vector (watched flag + normalized rating per anime)
- **Encoder**: Projects to 256-dim bottleneck with noise injection
- **Decoder**: Two heads — watch prediction (BCE) + rating prediction (Huber)
- **Loss**: Uncertainty-weighted combination (learnable per-head weights)

**Hardware**: T4 GPU (~16GB VRAM)

**Requires**: `02_preprocessing.ipynb` (needs `cf_ratings.npz` and `cf_anime_index.json`).

In [1]:
!pip install -q torch scipy tqdm

In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy import sparse
from pathlib import Path

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# Load CF data
cf_matrix = sparse.load_npz(DATA_DIR / "cf_ratings.npz")

with open(DATA_DIR / "cf_anime_index.json", "r") as f:
    anime_index = json.load(f)

n_users, n_anime = cf_matrix.shape
print(f"Rating matrix: {n_users:,} users × {n_anime:,} anime")
print(f"Non-zero ratings: {cf_matrix.nnz:,}")
print(f"Density: {cf_matrix.nnz / (n_users * n_anime):.4%}")

Rating matrix: 265,263 users × 12,127 anime
Non-zero ratings: 57,178,041
Density: 1.7775%


## Model Architecture

Following Sprout's design:
- Input: `anime_count * 2` (one slot for watched flag, one for normalized rating)
- Encoder: Dense → 1024 (Swish) → 256 bottleneck + Gaussian noise
- Decoder watch head: 256 → 1024 (Swish) → anime_count → Sigmoid
- Decoder rating head: 256 → 1024 (Swish) → anime_count
- Learnable uncertainty parameters for loss weighting

In [4]:
class AnimeCFAutoencoder(nn.Module):
    def __init__(self, n_anime, bottleneck_dim=256, hidden_dim=1024, noise_std=0.1):
        super().__init__()
        self.n_anime = n_anime
        self.noise_std = noise_std
        input_dim = n_anime * 2  # watched flags + normalized ratings

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),  # Swish activation
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, bottleneck_dim)
        )

        # Decoder - watch prediction head
        self.watch_decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_anime)
        )

        # Decoder - rating prediction head
        self.rating_decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, n_anime)
        )

        # Learnable uncertainty parameters (Sprout-style)
        self.log_var_watch = nn.Parameter(torch.zeros(1))
        self.log_var_rating = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # Encode
        z = self.encoder(x)

        # Add noise during training (denoising autoencoder)
        if self.training:
            z = z + torch.randn_like(z) * self.noise_std

        # Decode
        watch_logits = self.watch_decoder(z)
        rating_pred = self.rating_decoder(z)

        return watch_logits, rating_pred

    def get_loss(
        self,
        watch_logits,
        rating_pred,
        watch_target,
        rating_target,
        rating_mask,
        item_positive_weights=None,
        weak_negative_weight=0.1,
    ):
        """Uncertainty-weighted multi-task loss with weak negatives + long-tail weighting."""
        pos_w = None

        # Watch loss (weighted BCE):
        # - watched items use long-tail weights (>=1)
        # - unwatched items are weak negatives (<1)
        if item_positive_weights is not None:
            pos_w = item_positive_weights
            if pos_w.dim() == 1:
                pos_w = pos_w.unsqueeze(0)
            if pos_w.shape[0] != watch_target.shape[0]:
                pos_w = pos_w.expand(watch_target.shape[0], -1)

            neg_w = torch.full_like(watch_target, float(weak_negative_weight))
            watch_weights = torch.where(watch_target > 0.5, pos_w, neg_w)

            watch_loss_raw = F.binary_cross_entropy_with_logits(
                watch_logits,
                watch_target,
                reduction="none",
            )
            watch_loss = (watch_loss_raw * watch_weights).sum() / watch_weights.sum().clamp_min(1e-8)
        else:
            watch_loss = F.binary_cross_entropy_with_logits(watch_logits, watch_target)

        # Rating loss (Huber, only for watched anime)
        if rating_mask.sum() > 0:
            rating_loss_raw = F.huber_loss(
                rating_pred[rating_mask],
                rating_target[rating_mask],
                delta=1.0,
                reduction="none",
            )
            if pos_w is not None:
                rating_weights = pos_w[rating_mask]
                rating_loss = (rating_loss_raw * rating_weights).sum() / rating_weights.sum().clamp_min(1e-8)
            else:
                rating_loss = rating_loss_raw.mean()
        else:
            rating_loss = torch.tensor(0.0, device=watch_logits.device)

        # Uncertainty weighting: L = (1/2sigma^2) * loss + log(sigma)
        precision_watch = torch.exp(-self.log_var_watch)
        precision_rating = torch.exp(-self.log_var_rating)

        total_loss = (
            0.5 * precision_watch * watch_loss + 0.5 * self.log_var_watch +
            0.5 * precision_rating * rating_loss + 0.5 * self.log_var_rating
        )

        return total_loss, watch_loss.item(), rating_loss.item()


model = AnimeCFAutoencoder(n_anime).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"Input dim: {n_anime * 2:,}, Bottleneck: 256, Output: {n_anime:,} per head")



Model parameters: 50,486,208 (50.5M)
Input dim: 24,254, Bottleneck: 256, Output: 12,127 per head


In [5]:
# Import from a real module so Windows multiprocessing workers can resolve the class.
import sys
from pathlib import Path
if not Path("training_datasets.py").exists() and Path("notebooks/training_datasets.py").exists():
    sys.path.append(str(Path("notebooks").resolve()))
from training_datasets import CFDataset

# Create dataset and split
full_dataset = CFDataset(
    cf_matrix,
    dropout_range=(0.45, 0.85),
    min_kept_items=2,
    long_tail_alpha=0.35,
    max_pos_weight=4.0,
)

item_positive_weights = torch.tensor(full_dataset.item_positive_weights, dtype=torch.float32, device=device)
print(
    f"Item weight stats: min={item_positive_weights.min().item():.3f}, "
    f"mean={item_positive_weights.mean().item():.3f}, max={item_positive_weights.max().item():.3f}"
)

train_size = int(0.95 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Train: {len(train_dataset):,}, Val: {len(val_dataset):,}")



Item weight stats: min=1.000, mean=1.319, max=2.534
Train: 251,999, Val: 13,264


In [6]:
# Hyperparameters
import os

BATCH_SIZE = 256
EPOCHS = 80
LR = 1e-3
WEAK_NEGATIVE_WEIGHT = 0.08

NUM_WORKERS = 0 if os.name == "nt" else 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, verbose=True)

print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LR}")
print(f"Weak negative weight: {WEAK_NEGATIVE_WEIGHT}")
print(f"Steps per epoch: {len(train_loader):,}")



Epochs: 80, Batch size: 256, LR: 0.001
Weak negative weight: 0.08
Steps per epoch: 985


C:\Users\jeddh\Projects\animetracker\notebooks\.venv311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [8]:
# Training loop
best_val_loss = float("inf")
cf_output_dir = MODEL_DIR / "anime_cf"
cf_output_dir.mkdir(exist_ok=True)

print("Starting CF training...")
for epoch in tqdm(range(EPOCHS), desc="CF epochs"):
    # Train
    model.train()
    train_losses = []
    train_watch_losses = []
    train_rating_losses = []

    for input_vec, watch_target, rating_target, rating_mask in tqdm(train_loader, desc=f"Train epoch {epoch+1}", leave=False):
        input_vec = input_vec.to(device)
        watch_target = watch_target.to(device)
        rating_target = rating_target.to(device)
        rating_mask = rating_mask.to(device)

        optimizer.zero_grad()
        watch_logits, rating_pred = model(input_vec)
        loss, w_loss, r_loss = model.get_loss(
            watch_logits,
            rating_pred,
            watch_target,
            rating_target,
            rating_mask,
            item_positive_weights=item_positive_weights,
            weak_negative_weight=WEAK_NEGATIVE_WEIGHT,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())
        train_watch_losses.append(w_loss)
        train_rating_losses.append(r_loss)

    # Validate
    model.eval()
    val_losses = []
    with torch.no_grad():
        for input_vec, watch_target, rating_target, rating_mask in tqdm(val_loader, desc=f"Val epoch {epoch+1}", leave=False):
            input_vec = input_vec.to(device)
            watch_target = watch_target.to(device)
            rating_target = rating_target.to(device)
            rating_mask = rating_mask.to(device)

            watch_logits, rating_pred = model(input_vec)
            loss, _, _ = model.get_loss(
                watch_logits,
                rating_pred,
                watch_target,
                rating_target,
                rating_mask,
                item_positive_weights=item_positive_weights,
                weak_negative_weight=WEAK_NEGATIVE_WEIGHT,
            )
            val_losses.append(loss.item())

    avg_train = np.mean(train_losses)
    avg_val = np.mean(val_losses)
    scheduler.step(avg_val)

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            "model_state_dict": model.state_dict(),
            "n_anime": n_anime,
            "epoch": epoch
        }, cf_output_dir / "best_model.pt")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"Train: {avg_train:.4f} (watch={np.mean(train_watch_losses):.4f}, rating={np.mean(train_rating_losses):.4f}) | "
            f"Val: {avg_val:.4f} | "
            f"sigma_watch={torch.exp(model.log_var_watch/2).item():.3f}, sigma_rating={torch.exp(model.log_var_rating/2).item():.3f}"
        )

print(f"Training complete. Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {cf_output_dir / 'best_model.pt'}")



Starting CF training...


CF epochs:   0%|          | 0/80 [00:00<?, ?it/s]

Train epoch 1:   0%|          | 0/985 [00:00<?, ?it/s]

Val epoch 1:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 1/80 | Train: 0.0833 (watch=0.1842, rating=0.5018) | Val: -0.1456 | sigma_watch=0.622, sigma_rating=0.715


...

Train epoch 79:   0%|          | 0/985 [00:00<?, ?it/s]

Val epoch 79:   0%|          | 0/52 [00:00<?, ?it/s]

Train epoch 80:   0%|          | 0/985 [00:00<?, ?it/s]

Val epoch 80:   0%|          | 0/52 [00:00<?, ?it/s]

Epoch 80/80 | Train: -0.4675 (watch=0.1281, rating=0.4148) | Val: -0.4540 | sigma_watch=0.358, sigma_rating=0.646
Training complete. Best val loss: -0.4546
Model saved to: models\anime_cf\best_model.pt


## Quick Validation

Test the model by predicting ratings for a few users.

In [9]:
import pandas as pd
# Load best model
checkpoint = torch.load(cf_output_dir / "best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Build anime ID lookup
idx_to_info = {}
for entry in anime_index:
    idx_to_info[entry["idx"]] = {
        "mal_id": entry["mal_id"],
        "anilist_id": entry.get("anilist_id")
    }

# Load anime titles from Kaggle for display
KAGGLE_INPUT = Path("/kaggle/input/anime-recommendation-database-2020")
if not KAGGLE_INPUT.exists():
    KAGGLE_INPUT = DATA_DIR / "kaggle"
anime_df = pd.read_csv(KAGGLE_INPUT / "anime.csv")
mal_id_to_name = dict(zip(anime_df["MAL_ID"], anime_df["Name"]))

C:\Users\jeddh\AppData\Local\Temp\ipykernel_60448\4017152506.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(cf_output_dir / "best_model.pt", map

In [10]:
# Pick a few random users and show their top predictions vs actual ratings
test_user_indices = np.random.choice(n_users, 3, replace=False)

for user_idx in test_user_indices:
    row = cf_matrix.getrow(user_idx).toarray().flatten()
    watched = (row != 0).astype(np.float32)
    ratings = row.astype(np.float32)

    # Full input (no dropout for inference)
    input_vec = np.concatenate([watched, ratings])
    input_tensor = torch.FloatTensor(input_vec).unsqueeze(0).to(device)

    with torch.no_grad():
        watch_logits, rating_pred = model(input_tensor)

    watch_probs = torch.sigmoid(watch_logits).cpu().numpy().flatten()
    pred_ratings = rating_pred.cpu().numpy().flatten()

    # Get top predictions for unwatched anime
    unwatched_mask = watched == 0
    scores = watch_probs * (pred_ratings + 5)  # Denormalize roughly
    scores[~unwatched_mask.astype(bool)] = -1  # Exclude watched

    top_indices = np.argsort(scores)[::-1][:10]

    print(f"\n{'='*60}")
    print(f"User {user_idx} | Watched: {int(watched.sum())} anime")
    print(f"{'='*60}")

    # Show some of their actual watched anime
    watched_indices = np.where(watched > 0)[0]
    print("\nSample watched (top rated):")
    watched_with_scores = [(i, ratings[i]) for i in watched_indices]
    watched_with_scores.sort(key=lambda x: x[1], reverse=True)
    for idx, score in watched_with_scores[:5]:
        info = idx_to_info.get(idx, {})
        name = mal_id_to_name.get(info.get("mal_id"), f"MAL#{info.get('mal_id', '?')}")
        print(f"  {name}: {score:+.1f} (raw)")

    print("\nTop 10 predictions:")
    for rank, idx in enumerate(top_indices, 1):
        info = idx_to_info.get(idx, {})
        name = mal_id_to_name.get(info.get("mal_id"), f"MAL#{info.get('mal_id', '?')}")
        print(f"  {rank}. {name} (watch_prob={watch_probs[idx]:.3f}, pred_rating={pred_ratings[idx]:+.2f})")


User 196147 | Watched: 84 anime

Sample watched (top rated):
  Death Note: +2.5 (raw)
  Shingeki no Kyojin: +2.5 (raw)
  Shingeki no Kyojin Season 2: +2.5 (raw)
  Kimi no Na wa.: +2.5 (raw)
  Shingeki no Kyojin Season 3: +2.5 (raw)

Top 10 predictions:
  1. Kiseijuu: Sei no Kakuritsu (watch_prob=0.977, pred_rating=+1.65)
  2. Mob Psycho 100 II (watch_prob=0.937, pred_rating=+1.77)
  3. Mob Psycho 100 (watch_prob=0.985, pred_rating=+1.43)
  4. Hunter x Hunter (2011) (watch_prob=0.925, pred_rating=+1.73)
  5. Yakusoku no Neverland (watch_prob=0.918, pred_rating=+1.38)
  6. JoJo no Kimyou na Bouken Part 5: Ougon no Kaze (watch_prob=0.959, pred_rating=+1.10)
  7. Gantz:O (watch_prob=0.928, pred_rating=+1.14)
  8. Berserk: Ougon Jidai-hen III - Kourin (watch_prob=0.862, pred_rating=+1.52)
  9. Shingeki no Kyojin Movie 1: Guren no Yumiya (watch_prob=0.952, pred_rating=+0.87)
  10. Sen to Chihiro no Kamikakushi (watch_prob=0.878, pred_rating=+1.33)

User 123161 | Watched: 23 anime

Sample wa

In [11]:
# Save anime index mapping for the sidecar
print(f"\n✓ CF model saved to: {cf_output_dir / 'best_model.pt'}")
print(f"  Architecture: {n_anime*2} → 1024 → 256 → 1024 → {n_anime} (×2 heads)")
model_size = (cf_output_dir / 'best_model.pt').stat().st_size / 1e6
print(f"  Checkpoint size: {model_size:.1f} MB")
print(f"\nProceed to 05_export.ipynb")


✓ CF model saved to: models\anime_cf\best_model.pt
  Architecture: 24254 → 1024 → 256 → 1024 → 12127 (×2 heads)
  Checkpoint size: 202.0 MB

Proceed to 05_export.ipynb
